# MedImageInsight: Zero-Shot H&E Evaluation on Azure ML

Submit a two-component Azure ML pipeline from your computer to rank H&E images using Responder and Non-Responder prompts.
The workflow calculates ROC/AUC without training a model.

| Step | Script | Action | Inference requests |
|---|---|---|---|
| 1. Embedding | [scripts/embed_tiles.py](scripts/embed_tiles.py) | Tile slides at a consistent scale and embed selected tiles | Image batches |
| 2. Scoring | [scripts/score_zero_shot.py](scripts/score_zero_shot.py) | Compare saved embeddings with two prompts and calculate metrics | One text request |

The embedding step selects files from the cohort, but response labels do not influence the embeddings.
Section 6 reuses the saved embeddings with different prompts, without sending image tiles again.

## Prerequisites

- Open this folder as the project root. Select a Python 3.12 kernel with [requirements.txt](requirements.txt) installed.
- Copy [sample.env](sample.env) to `.env` beside this notebook. Configure your own workspace, CPU cluster, model deployment, and tenant.
- Configure an existing `uri_folder` slide asset through `HNE_DATA_NAME` and optionally `HNE_DATA_VERSION`. This notebook does not provision the workspace, cluster, or slide asset.
- Place your private label sheet at `data/labels.csv`, or configure `LABELS_CSV` in `.env`. The required columns are documented in [README.md](README.md#label-sheet).
- Submission uses your Azure CLI identity; jobs use the compute's managed identity, which needs permission to call your Foundry deployment.
- The API workflow is introduced in [00_local_zero_shot.ipynb](00_local_zero_shot.ipynb).

## Evaluation Scope

Each row is one image. Images are not aggregated or deduplicated by subject; confirm that this is appropriate for your dataset.
Class imbalance and related images can affect evaluation. Consider ROC-AUC and balanced accuracy alongside class-specific metrics.
No customer images, cohort statistics, or measured results are included in this notebook.

**For research only, not diagnosis or clinical decision-making.** Cloud jobs and inference can incur charges.
Keep data and generated artifacts private, and clear all notebook outputs before publication.
Offline tests use synthetic data and mocked services: `python scripts/test_zero_shot.py`.

In [ ]:
import os

from azure.ai.ml import Input, MLClient, dsl, load_component
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data, Environment, ManagedIdentityConfiguration
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import display

import sys
sys.path.insert(0, "scripts")
from mii_client import ensure_azure_cli_path, resource_endpoint

load_dotenv(".env", override=True)

SUBSCRIPTION_ID = os.environ["SUBSCRIPTION_ID"]
RESOURCE_GROUP = os.environ["RESOURCE_GROUP"]
AML_WORKSPACE_NAME = os.environ["AML_WORKSPACE_NAME"]
TENANT_ID = os.environ["TENANT_ID"]

CPU_CLUSTER = os.environ["AML_CPU_CLUSTER"]

# Foundry settings are passed to jobs as strings.
MODEL = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME") or "MedImageInsight-Premium"
ENDPOINT = resource_endpoint(os.environ["AZURE_AI_PROJECT_ENDPOINT"])

# Set only for a user-assigned identity; leave empty for a system-assigned identity.
MANAGED_IDENTITY_CLIENT_ID = os.getenv("AML_MANAGED_IDENTITY_CLIENT_ID", "")

ensure_azure_cli_path()
credential = AzureCliCredential(tenant_id=TENANT_ID, process_timeout=120)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, AML_WORKSPACE_NAME)

print(ml_client.workspace_name, "|", CPU_CLUSTER, "|", MODEL)
print(ENDPOINT)

## 1. Register the Job Environment

Register the compute environment defined in [environments/mii-embed-env.yml](environments/mii-embed-env.yml).
This definition is separate from your local Python environment.

OpenSlide is installed through conda; `openslide-python` is the Python binding for the native library.
The definition includes a pixman version constraint. Keep native-library requirements when customizing the environment.
Environment registration and the initial image build can take time.

In [ ]:
embed_env = Environment(
    name="mii-embed-env",
    description="OpenSlide tiling + MedImageInsight embedding client, CPU only",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    conda_file="environments/mii-embed-env.yml",
)
embed_env = ml_client.environments.create_or_update(embed_env)
print(embed_env.name, embed_env.version)

## 2. Select the Slide Data Asset

Set `HNE_DATA_NAME` in this folder's `.env` to an existing Azure ML `uri_folder` asset containing your authorized SVS files.
Set `HNE_DATA_VERSION` to pin a version, or leave it empty to resolve the latest version.
The notebook does not create the slide asset.

The pipeline mounts the selected folder read-only; this step does not download SVS files to the local computer.
Record the resolved asset name and version with your private experiment records.

In [ ]:
DATA_NAME = os.environ["HNE_DATA_NAME"]
DATA_VERSION = os.getenv("HNE_DATA_VERSION", "")

wsi_data = (ml_client.data.get(DATA_NAME, version=DATA_VERSION) if DATA_VERSION
            else ml_client.data.get(DATA_NAME, label="latest"))
if wsi_data.type != AssetTypes.URI_FOLDER:
    raise ValueError(f"{DATA_NAME} must be a uri_folder asset containing SVS files.")

print(wsi_data.name, wsi_data.version, wsi_data.type)

## 3. Build the Image Cohort

Place your private label sheet at `data/labels.csv`, or configure `LABELS_CSV` in the local `.env`.
Required CSV columns are `H&E Images` (image path or file name) and `Responder/Not Responder` (response label).
Image file names must match the slides in your registered data asset.

[scripts/cohort.py](scripts/cohort.py) creates `outputs/cohort.csv` with only `file_name` and `label`.
Unlabeled rows are excluded before embedding and scoring. The next cell registers this generated file as a pipeline input.

**This is data minimization, not anonymization.** File names can contain patient or specimen identifiers, and response labels are clinical information.
Keep both the source sheet and generated cohort private. The `data/` and `outputs/` directories are ignored by Git.

Rows remain image-level records without subject aggregation or label reassignment. Confirm that this evaluation unit is appropriate for your data.

In [ ]:
from collections import Counter

from cohort import labeled_only, read_cohort, write_cohort

LABELS_CSV = os.getenv("LABELS_CSV") or "data/labels.csv"
COHORT_CSV = "outputs/cohort.csv"
os.makedirs("outputs", exist_ok=True)

rows = read_cohort(LABELS_CSV)
labeled = labeled_only(rows)
write_cohort(COHORT_CSV, labeled)

print(f"{len(labeled)} labeled images out of {len(rows)} rows ->", COHORT_CSV)
print(dict(Counter(row["label"] for row in labeled)))

In [ ]:
cohort_data = ml_client.data.create_or_update(
    Data(
        name="mii_cohort",
        description="Image-level cohort for zero-shot scoring: file_name -> Responder / Non-Responder",
        path=COHORT_CSV,
        type=AssetTypes.URI_FILE,
    )
)
print(cohort_data.name, cohort_data.version)

## 4. Build the Pipeline

[component/tile_embedding.yml](component/tile_embedding.yml) and [component/zero_shot_score.yml](component/zero_shot_score.yml)
define the scripts, inputs, and outputs. `load_component` loads these definitions for use in the Python pipeline.

### Tiling Parameters

| Parameter | Default | Meaning |
|---|---:|---|
| `target_mpp` | 0.5 | Requested microns per pixel |
| `tile_size` | 512 | Output tile width and height in pixels |
| `occupancy_threshold` | 0.20 | Tissue occupancy threshold for candidate tiles |
| `max_tiles` | 32 | Maximum selected tiles per slide |
| `batch_size` | 8 | Maximum tiles per image request |

`target_mpp` is a preprocessing choice, not a validated model requirement. The tiler reads each slide's metadata, selects a suitable pyramid level, and resizes the region.
Candidate tiles are sampled at evenly spaced indices in row-major order, not ranked by tissue density.
**This does not cover all tissue.** Record parameter changes as separate experimental conditions.

### Request Estimate

The upper-bound image-request estimate is `ceil(max_tiles / batch_size) * number_of_images`, plus one text request for scoring.
Slides with fewer candidate tiles can require fewer image requests. Review the estimate before submission.

In [ ]:
import math

TARGET_MPP = 0.5
TILE_SIZE = 512
OCCUPANCY_THRESHOLD = 0.20
MAX_TILES = 32
BATCH_SIZE = 8

# These prompts define the classifier; the order is Non-Responder, Responder.
PROMPT_NEGATIVE = "An H&E histopathology image of a lesion with no response to treatment."
PROMPT_POSITIVE = "An H&E histopathology image of a lesion with response to treatment."

n_images = len(labeled)
image_calls = math.ceil(MAX_TILES / BATCH_SIZE) * n_images
print(f"{n_images} images x up to {MAX_TILES} tiles = up to {n_images * MAX_TILES} tiles")
print(f"API requests: up to {image_calls} image batches + 1 text request = up to {image_calls + 1}")

In [ ]:
tile_embedding_component = load_component(source="component/tile_embedding.yml")
zero_shot_score_component = load_component(source="component/zero_shot_score.yml")


@dsl.pipeline(description="MedImageInsight zero-shot: tile embedding then prompt scoring")
def pipeline_mii_zero_shot(images_dir, cohort_csv, endpoint, model, prompt_negative,
                           prompt_positive, target_mpp, tile_size, occupancy_threshold,
                           max_tiles, batch_size):
    embedding_job = tile_embedding_component(
        images_dir=images_dir,
        cohort_csv=cohort_csv,
        endpoint=endpoint,
        model=model,
        target_mpp=target_mpp,
        tile_size=tile_size,
        occupancy_threshold=occupancy_threshold,
        max_tiles=max_tiles,
        batch_size=batch_size,
        # Omit the optional input with None rather than passing an empty string.
        managed_identity_client_id=MANAGED_IDENTITY_CLIENT_ID or None,
    )

    scoring_job = zero_shot_score_component(
        embeddings_dir=embedding_job.outputs.output_dir,
        cohort_csv=cohort_csv,
        endpoint=endpoint,
        model=model,
        prompt_negative=prompt_negative,
        prompt_positive=prompt_positive,
        managed_identity_client_id=MANAGED_IDENTITY_CLIENT_ID or None,
    )

    # Both steps run on CPU and use the compute identity for Foundry requests.
    # Assign settings explicitly to retain meaningful pipeline node names.
    embedding_job.compute = CPU_CLUSTER
    embedding_job.identity = ManagedIdentityConfiguration(client_id=MANAGED_IDENTITY_CLIENT_ID or None)
    embedding_job.environment_variables = {"PYTHONUNBUFFERED": "1"}

    scoring_job.compute = CPU_CLUSTER
    scoring_job.identity = ManagedIdentityConfiguration(client_id=MANAGED_IDENTITY_CLIENT_ID or None)
    scoring_job.environment_variables = {"PYTHONUNBUFFERED": "1"}

    return {
        "embeddings": embedding_job.outputs.output_dir,
        "evaluation": scoring_job.outputs.output_dir,
    }

### Submit the Pipeline

**The following cell submits a cloud job and can incur compute and inference charges.**
Review the parameters and request estimate first.

Record the returned job name privately. The following `stream()` cell displays logs; stopping log streaming does not cancel the cloud job.
After a kernel restart, use the existing job name to inspect status and retrieve results instead of blindly submitting again.

In [ ]:
pipeline = pipeline_mii_zero_shot(
    images_dir=Input(type=AssetTypes.URI_FOLDER, path=wsi_data.id, mode="ro_mount"),
    cohort_csv=Input(type=AssetTypes.URI_FILE, path=cohort_data.id),
    endpoint=ENDPOINT,
    model=MODEL,
    prompt_negative=PROMPT_NEGATIVE,
    prompt_positive=PROMPT_POSITIVE,
    target_mpp=TARGET_MPP,
    tile_size=TILE_SIZE,
    occupancy_threshold=OCCUPANCY_THRESHOLD,
    max_tiles=MAX_TILES,
    batch_size=BATCH_SIZE,
)

pipeline_job = ml_client.jobs.create_or_update(pipeline, experiment_name="medimageinsight-zero-shot")
print(pipeline_job.name)
print(pipeline_job.studio_url)

In [ ]:
ml_client.jobs.stream(pipeline_job.name)

## 5. Download and Inspect Results

After the job completes, download the scoring output named `evaluation`. This does not download the original SVS files.
`metrics.json` contains aggregate metrics and the confusion matrix; `predictions.csv` contains image-level scores.
[scripts/plot_zero_shot.py](scripts/plot_zero_shot.py) renders saved results locally, so plots can be regenerated without inference.

Responder is the positive class. The image score is:

```text
mean_tile_cosine(positive_prompt) - mean_tile_cosine(negative_prompt)
```

This is a ranking score, not a probability. The confusion matrix uses the fixed rule `score > 0`.
No training, threshold tuning, or post-hoc sign reversal is performed.
Treat the downloaded results and embedded plots as private derived data, and clear outputs before publication.

In [ ]:
# After a kernel restart, set JOB_NAME to your previously submitted job name.
JOB_NAME = pipeline_job.name
# JOB_NAME = "<your-existing-job-name>"
print(JOB_NAME, ml_client.jobs.get(JOB_NAME).status)

In [ ]:
import json

import pandas as pd

ml_client.jobs.download(JOB_NAME, download_path="./outputs", output_name="evaluation")

result_dir = "./outputs/named-outputs/evaluation"
metrics = json.loads(open(f"{result_dir}/metrics.json", encoding="utf-8").read())
print(json.dumps({key: value for key, value in metrics.items() if key != "prompts"}, indent=2))

predictions = pd.read_csv(f"{result_dir}/predictions.csv", float_precision="round_trip")
display(predictions[["image_id", "label", "predicted_label", "score", "tile_count"]])

In [ ]:
# Plot ROC, the zero-threshold confusion matrix, and score distributions.
from plot_zero_shot import report

# Assign the figure to avoid duplicate rendering by the inline backend.
figure = report(result_dir)

### Inspect Resolution and Sampling

`tiles.json` records the resolution used for each slide. Integer source-pixel dimensions can cause a small difference
between `effective_mpp` and `target_mpp`. The tiler checks the relative difference against `MPP_TOLERANCE` (1%)
and rejects unsupported or inconsistent settings during the job. The following cell checks the saved metadata.

Also inspect `candidate_tiles`, `selected_tiles`, and `sampling_fraction`. A small selected fraction means that only
a limited portion of candidate tissue contributed to the image-level score. No dataset-specific measurements are shown here.

In [ ]:
from tile_wsi import MPP_TOLERANCE

ml_client.jobs.download(JOB_NAME, download_path="./outputs", output_name="embeddings")

tiles = json.loads(open("./outputs/named-outputs/embeddings/tiles.json", encoding="utf-8").read())
display(pd.DataFrame(tiles)[["image_id", "base_mpp", "read_level", "effective_mpp",
                             "candidate_tiles", "selected_tiles", "sampling_fraction"]])

drift = max(abs(row["effective_mpp"] - row["target_mpp"]) / row["target_mpp"] for row in tiles)
print(f"target_mpp: {sorted({row['target_mpp'] for row in tiles})}")
print(f"Maximum drift: {drift:.3%} (tolerance: {MPP_TOLERANCE:.0%}) ->",
      "OK" if drift <= MPP_TOLERANCE else "NG")

## 6. Re-Score Saved Embeddings with Different Prompts

Register the saved tile embeddings as a data asset, then run only the scoring component with different text prompts.
Image tiles are not sent again. Text embedding and compute can still incur charges.
Confirm that the selected cohort asset describes exactly the images in the saved embeddings.

**Selecting the best prompt using these same evaluation labels is not independent validation.**
Keep a record of every tested condition and use independent data for any later performance claims.
Retain comparison results privately rather than embedding them in a public notebook.

In [ ]:
embeddings_data = ml_client.data.create_or_update(
    Data(
        name="mii_tile_embeddings",
        description=f"Tile embeddings from job {JOB_NAME} (target_mpp={TARGET_MPP}, max_tiles={MAX_TILES})",
        path="./outputs/named-outputs/embeddings",
        type=AssetTypes.URI_FOLDER,
    )
)
print(embeddings_data.name, embeddings_data.version)

In [ ]:
@dsl.pipeline(description="MedImageInsight zero-shot: re-score saved embeddings with new prompts")
def pipeline_mii_rescore(embeddings_dir, cohort_csv, endpoint, model,
                         prompt_negative, prompt_positive):
    scoring_job = zero_shot_score_component(
        embeddings_dir=embeddings_dir,
        cohort_csv=cohort_csv,
        endpoint=endpoint,
        model=model,
        prompt_negative=prompt_negative,
        prompt_positive=prompt_positive,
        managed_identity_client_id=MANAGED_IDENTITY_CLIENT_ID or None,
    )
    scoring_job.compute = CPU_CLUSTER
    scoring_job.identity = ManagedIdentityConfiguration(client_id=MANAGED_IDENTITY_CLIENT_ID or None)
    return {"evaluation": scoring_job.outputs.output_dir}

In [ ]:
rescore = pipeline_mii_rescore(
    embeddings_dir=Input(type=AssetTypes.URI_FOLDER, path="mii_tile_embeddings@latest"),
    cohort_csv=Input(type=AssetTypes.URI_FILE, path="mii_cohort@latest"),
    endpoint=ENDPOINT,
    model=MODEL,
    prompt_negative="An H&E histopathology image showing viable residual tumor.",
    prompt_positive="An H&E histopathology image showing tumor regression with fibrosis and necrosis.",
)

rescore_job = ml_client.jobs.create_or_update(rescore, experiment_name="medimageinsight-zero-shot")
print(rescore_job.name, rescore_job.studio_url)
ml_client.jobs.stream(rescore_job.name)

In [ ]:
ml_client.jobs.download(rescore_job.name, download_path="./outputs/rescore", output_name="evaluation")

rescore_dir = "./outputs/rescore/named-outputs/evaluation"
rescore_metrics = json.loads(open(f"{rescore_dir}/metrics.json", encoding="utf-8").read())
print("prompts:", json.dumps(rescore_metrics["prompts"], indent=2, ensure_ascii=False))
print(f"AUC {metrics['roc_auc']:.3f} -> {rescore_metrics['roc_auc']:.3f}")

figure = report(rescore_dir)